# RQ2: Base temperature and threshold validation (2016 to 2026)

This notebook selects the base temperature (T_base) of the GDD onset model and validates the threshold T* leave-one-year-out with KNMI temperatures and observed Betula onsets in Helmond and Leiden.
GDD = max(0, (TX + TN) / 2 − T_base), accumulated from 1 March. The onset is the day on which 5 % of the seasonal pollen sum is reached.

**Data (not included in this repository):** KNMI daily data of Eindhoven (370) and Voorschoten (215), and the pollen counts of Helmond (Elkerliek) and Leiden (LUMC).

## Step 1: Setup
Load the packages and set the file paths and settings.

In [ ]:
import warnings
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
from IPython.display import display

warnings.filterwarnings('ignore')

DATA_DIR = Path('data')
OUT_DIR = Path('output_tbase')
OUT_DIR.mkdir(exist_ok=True)
KNMI_FILES = {'Helmond': DATA_DIR / 'knmi_eindhoven.txt',   # station 370
              'Leiden': DATA_DIR / 'knmi_voorschoten.txt'}  # station 215
HELMOND_POLLEN = DATA_DIR / 'pollen_helmond_all_years.xlsx'
LEIDEN_POLLEN = DATA_DIR / 'pollen_leiden.xlsx'

T_BASES = [0, 1, 2, 3, 3.5, 4, 5, 6, 7, 8, 9, 10]   # °C
D0_MONTH_DAY = (3, 1)     # start of accumulation: 1 March
ONSET_FRACTION = 0.05     # 5 % of the seasonal sum
SEASON_END = (5, 31)      # accumulate until 31 May
MAX_MISSING_T = 3         # skip a year with more than 3 missing temperature days in March to May
FIRST_YEAR = 2016         # main analysis 2016 to 2026 for both stations

## Step 2: KNMI daily temperature
Read the KNMI daily files and compute the daily mean temperature as (TX + TN) / 2.

In [ ]:
def read_knmi(path):
    # KNMI daily file, columns YYYYMMDD, TG, TN, TX in 0.1 °C
    rows = []
    with open(path, encoding='latin-1') as f:
        for line in f:
            s = line.strip()
            if not s or not s[0].isdigit():
                continue
            parts = [x.strip() for x in s.split(',')]
            rows.append((parts[1], parts[11], parts[12], parts[14]))
    k = pd.DataFrame(rows, columns=['date', 'TG', 'TN', 'TX'])
    k['date'] = pd.to_datetime(k['date'], format='%Y%m%d')
    for c in ['TG', 'TN', 'TX']:
        k[c] = pd.to_numeric(k[c], errors='coerce') / 10.0
    k = k.set_index('date').sort_index()
    k['Tmean'] = (k['TX'] + k['TN']) / 2.0
    return k

KNMI = {loc: read_knmi(p) for loc, p in KNMI_FILES.items()}
for loc, k in KNMI.items():
    print(f"{loc:8s} {k.index.min().date()} to {k.index.max().date()}, missing Tmean: {k['Tmean'].isna().sum()}")

## Step 3: Pollen counts
Read the Betula counts per year, taking the year of each Helmond sheet from the sheet name because the date headers come from older templates.

In [ ]:
def read_helmond(path):
    xl = pd.ExcelFile(path)
    out = {}
    for sh in xl.sheet_names:
        if not sh.strip().isdigit():
            continue
        year = int(sh.strip())
        d = pd.read_excel(xl, sheet_name=sh, header=None)
        col0 = d.iloc[:, 0].astype(str).str.strip()
        rows_betula = np.where(col0 == 'Betula')[0]
        if len(rows_betula) == 0:
            print(f'{year}: no Betula row, skipped')
            continue
        # header row = the row (0 to 2) with the most dates
        best_r, best_dates = None, None
        for r in range(3):
            v = pd.to_datetime(d.iloc[r, 1:], errors='coerce')
            if best_dates is None or v.notna().sum() > best_dates.notna().sum():
                best_r, best_dates = r, v
        values = pd.to_numeric(d.iloc[rows_betula[0], 1:], errors='coerce')
        rec, seen_jan = [], False
        for hd, val in zip(best_dates, values):
            if pd.isna(hd):
                continue
            m, day = hd.month, hd.day
            if m == 1:
                seen_jan = True
            y = year - 1 if (m == 12 and not seen_jan) else year   # December columns belong to the previous year
            try:
                rec.append((pd.Timestamp(dt.date(y, m, day)), val))
            except ValueError:   # 29 February in a non-leap year
                pass
        s = pd.Series(dict(rec)).sort_index()
        out[year] = s[~s.index.duplicated()]
    return out

def read_leiden(path):
    d = pd.read_excel(path)
    d['date'] = pd.to_datetime(d['date'])
    s = d.set_index('date')['Betula']
    return {y: g for y, g in s.groupby(s.index.year)}

POLLEN = {'Helmond': read_helmond(HELMOND_POLLEN), 'Leiden': read_leiden(LEIDEN_POLLEN)}
print({loc: sorted(p) for loc, p in POLLEN.items()})

## Step 4: Observed onset per year
Determine the onset per station and year as the first day on which 5 % of the seasonal sum is reached.

In [ ]:
def onset_5pct(s, frac=ONSET_FRACTION):
    s = s.fillna(0)
    cum = s.cumsum()
    total = s.sum()
    return cum[cum >= frac * total].index[0], total

rows = []
for loc, years in POLLEN.items():
    for y, s in sorted(years.items()):
        on, tot = onset_5pct(s)
        rows.append({'location': loc, 'year': y, 'onset': on.date(), 'doy': on.dayofyear,
                     'seasonal_total': int(tot), 'days_with_count': int(s.notna().sum())})
onsets_all = pd.DataFrame(rows)   # all years (Helmond from 1997)
onsets = onsets_all if FIRST_YEAR is None else onsets_all[onsets_all.year >= FIRST_YEAR].reset_index(drop=True)
print('Seasons per station:', onsets.groupby('location').size().to_dict())
display(onsets)

## Step 5: Leave-one-year-out validation
For each T_base, compute T* per year and predict each year with the mean T* of the other years.

In [ ]:
def season_gdd(tmean, year, t_base):
    # cumulative GDD from 1 March to 31 May, None if too many days are missing
    start = pd.Timestamp(year, *D0_MONTH_DAY)
    end = pd.Timestamp(year, *SEASON_END)
    seg = tmean.loc[start:end]
    if len(seg) == 0 or seg.isna().sum() > MAX_MISSING_T:
        return None
    seg = seg.interpolate(limit_direction='both')
    return np.maximum(seg - t_base, 0.0).cumsum()

loyo_rows, tstar_rows = [], []
for loc in POLLEN:
    tmean = KNMI[loc]['Tmean']
    obs = onsets[onsets.location == loc].set_index('year')['onset']
    for tb in T_BASES:
        cum, tstar = {}, {}
        for y, on in obs.items():
            c = season_gdd(tmean, y, tb)
            if c is None or pd.Timestamp(on) not in c.index:
                continue
            cum[y] = c
            tstar[y] = c.loc[pd.Timestamp(on)]
            tstar_rows.append({'location': loc, 'T_base': tb, 'year': y, 'T_star': tstar[y]})
        for y in tstar:
            t_other = np.mean([v for yy, v in tstar.items() if yy != y])
            c = cum[y]
            hit = c[c >= t_other]
            pred = hit.index[0] if len(hit) else pd.NaT
            err = (pred - pd.Timestamp(obs[y])).days if len(hit) else np.nan
            loyo_rows.append({'location': loc, 'T_base': tb, 'year': y,
                              'T_star_loyo': t_other, 'observed': obs[y],
                              'predicted': None if pd.isna(pred) else pred.date(), 'error_days': err})

tstar_df = pd.DataFrame(tstar_rows)
loyo = pd.DataFrame(loyo_rows)

summary = (loyo.groupby(['location', 'T_base'])
               .agg(n_years=('error_days', 'count'),
                    MAE=('error_days', lambda e: e.abs().mean()),
                    RMSE=('error_days', lambda e: np.sqrt((e ** 2).mean())),
                    bias=('error_days', 'mean'),
                    max_abs=('error_days', lambda e: e.abs().max()))
               .reset_index())
cv = (tstar_df.groupby(['location', 'T_base'])['T_star']
              .agg(T_star_mean='mean', T_star_CV=lambda v: v.std() / v.mean()).reset_index())
summary = summary.merge(cv, on=['location', 'T_base']).round(2)
display(summary)

## Step 6: MAE and stability of T* per T_base
Plot the leave-one-year-out MAE and the coefficient of variation of T* against T_base.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for loc, ls in [('Helmond', '-o'), ('Leiden', '-s')]:
    s = summary[summary.location == loc]
    axes[0].plot(s.T_base, s.MAE, ls, label=loc)
    axes[1].plot(s.T_base, s.T_star_CV, ls, label=loc)
axes[0].set(xlabel='T_base (°C)', ylabel='LOYO MAE (days)', title='Onset error over multiple years')
axes[1].set(xlabel='T_base (°C)', ylabel='CV of T*', title='Year-to-year stability of T*')
for ax in axes:
    ax.axvline(3.5, color='grey', ls=':', lw=1)
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## Step 7: Helmond with all seasons since 1997
Repeat the validation for Helmond with all 28 seasons and test for a trend in T* at 3.5 °C.

In [ ]:
tmean = KNMI['Helmond']['Tmean']
obs_all = onsets_all[onsets_all.location == 'Helmond'].set_index('year')['onset']
rows = []
for tb in T_BASES:
    cum, ts = {}, {}
    for y, on in obs_all.items():
        c = season_gdd(tmean, y, tb)
        if c is None: continue
        cum[y] = c; ts[y] = c.loc[pd.Timestamp(on)]
    err = []
    for y in ts:
        t_other = np.mean([v for k, v in ts.items() if k != y])
        hit = cum[y][cum[y] >= t_other]
        err.append(abs((hit.index[0] - pd.Timestamp(obs_all[y])).days))
    v = pd.Series(ts)
    rows.append({'T_base': tb, 'n_years': len(ts), 'MAE': np.mean(err), 'T_star_CV': v.std() / v.mean()})
h28 = pd.DataFrame(rows)
display(h28.round(2))

ts35 = {y: season_gdd(tmean, y, 3.5).loc[pd.Timestamp(on)] for y, on in obs_all.items() if season_gdd(tmean, y, 3.5) is not None}
r = stats.linregress(list(ts35), list(ts35.values()))
print(f'Trend in T* (3.5 °C), Helmond 1997 to 2026: {r.slope:.2f} °C-days per year, p = {r.pvalue:.2f}')

## Step 8: T* in 2026
Show T* of 2026 per T_base, which matches the in-sample threshold of the RQ2 forecast notebook.

In [ ]:
t26 = (tstar_df[tstar_df.year == 2026]
       .pivot(index='T_base', columns='location', values='T_star').round(1))
display(t26)

## Step 9: Plateau effect at high T_base
Show that at high T_base only a few days exceed T_base, so the cumulative GDD flattens and reaches T* before the onset even with observed temperatures.

In [ ]:
rows = []
for loc in POLLEN:
    tmean = KNMI[loc]['Tmean']
    on = pd.Timestamp(onsets[(onsets.location == loc) & (onsets.year == 2026)]['onset'].iloc[0])
    t = tmean.loc[pd.Timestamp(2026, *D0_MONTH_DAY):on]
    for tb in T_BASES:
        c = season_gdd(tmean, 2026, tb)
        ts = c.loc[on]
        first = c[c >= ts - 1e-9].index[0]
        rows.append({'location': loc, 'T_base': tb, 'T_star_2026': round(ts, 1),
                     'days_above_T_base': int((t > tb).sum()),
                     'GDD_per_day_last_7d': round(np.maximum(t.iloc[-7:] - tb, 0).mean(), 2),
                     'error_with_observed_temp': (first - on).days})
plateau = pd.DataFrame(rows)
display(plateau.pivot(index='T_base', columns='location',
                      values=['days_above_T_base', 'GDD_per_day_last_7d', 'error_with_observed_temp']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)
for ax, loc in zip(axes, POLLEN):
    tmean = KNMI[loc]['Tmean']
    on = pd.Timestamp(onsets[(onsets.location == loc) & (onsets.year == 2026)]['onset'].iloc[0])
    for tb in [3.5, 7, 10]:
        c = season_gdd(tmean, 2026, tb).loc[:on + pd.Timedelta(days=10)]
        ax.plot(c.index, c / c.loc[on], label=f'T_base {tb} °C')
    ax.axvline(on, color='purple', ls='--', lw=1.5, label='observed onset')
    ax.axhline(1, color='grey', ls=':', lw=1)
    ax.set(title=f'{loc} 2026: cumulative GDD scaled to T* (= 1)', ylabel='GDD / T*')
    ax.tick_params(axis='x', rotation=45); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / 'fig_gdd_scaled_to_tstar_2026.png', dpi=200)
plt.show()

## Step 10: Days above 10 °C
Count the days with a mean temperature above 10 °C between 28 February and the onset, for comparison with Picornell et al. (2019).

In [ ]:
rows = []
for loc in POLLEN:
    tmean = KNMI[loc]['Tmean']
    for y, on in onsets[onsets.location == loc].set_index('year')['onset'].items():
        seg = tmean.loc[pd.Timestamp(y, 2, 28):pd.Timestamp(on)]
        if seg.isna().sum() > MAX_MISSING_T:
            continue
        rows.append({'location': loc, 'year': y, 'days_above_10C': int((seg > 10).sum())})
d10 = pd.DataFrame(rows)
display(d10.groupby('location')['days_above_10C'].describe().round(1))
display(d10[d10.year == 2026])

## Step 11: MAE against T_base with benchmark
Plot the leave-one-year-out MAE per T_base together with the benchmark without temperature (mean onset of the other seasons).

In [ ]:
def days_since_d0(on, year):
    return (pd.Timestamp(on) - pd.Timestamp(year, *D0_MONTH_DAY)).days

def benchmark_mae(obs):
    # predict each year with the mean onset of the other years
    d = pd.Series({y: days_since_d0(on, y) for y, on in obs.items()})
    return float(np.mean([abs(d.drop(y).mean() - d[y]) for y in d.index]))

bench = {loc: benchmark_mae(onsets[onsets.location == loc].set_index('year')['onset']) for loc in POLLEN}
years_ok = [y for y in obs_all.index if season_gdd(KNMI['Helmond']['Tmean'], y, 3.5) is not None]
bench28 = benchmark_mae(obs_all.loc[years_ok])
h28['n'] = h28['n_years']
print('Benchmark (days):', {k: round(v, 2) for k, v in bench.items()}, '| Helmond 28 seasons:', round(bench28, 2))

COL = {'Helmond': '#E64A19', 'Leiden': '#388E3C'}
INK, INK2, GRID, BENCH = '#222222', '#555555', '#e3e3e3', '#7a7a7a'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.edgecolor': '#999999', 'axes.linewidth': 0.8,
                     'xtick.color': INK2, 'ytick.color': INK2, 'axes.labelcolor': INK, 'text.color': INK})
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.35), sharey=True)
for ax, loc in zip(axes, ['Helmond', 'Leiden']):
    d = summary[summary.location == loc].sort_values('T_base')
    ax.grid(True, color=GRID, lw=0.8, zorder=0); ax.set_axisbelow(True)
    ax.axvline(3.5, color='#444444', lw=1.1, ls=(0, (1.5, 2.5)), zorder=1)
    ax.text(3.62, 0.25, '3.5 °C', fontsize=8.2, color=INK2, va='bottom', ha='left')
    ax.axhline(bench[loc], color=BENCH, lw=1.5, ls=(0, (6, 3)), zorder=2)
    if loc == 'Helmond':
        ax.plot(h28.T_base, h28.MAE, color='#8E2A0B', lw=1.5, ls=(0, (4, 2)), marker='s', ms=4.6,
                mfc='white', mec='#8E2A0B', mew=1.2, zorder=3)
    ax.plot(d.T_base, d.MAE, color=COL[loc], lw=2.0, marker='o', ms=5.2, mfc=COL[loc], mec='white', mew=0.8, zorder=4)
    ax.set_title(loc, fontsize=11, fontweight='bold', color=INK, pad=7)
    ax.set_xlabel(r'$T_{\mathrm{base}}$ (°C)')
    ax.set_xlim(-0.35, 10.35); ax.set_xticks(range(0, 11))
    ax.set_ylim(0, 12.6); ax.set_yticks(range(0, 13, 2))
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    handles = [Line2D([0], [0], color=COL[loc], lw=2.0, marker='o', ms=5.2, mec='white', mew=0.8,
                      label='11 seasons (2016–2026)')]
    if loc == 'Helmond':
        handles.append(Line2D([0], [0], color='#8E2A0B', lw=1.5, ls=(0, (4, 2)), marker='s', ms=4.6, mfc='white',
                              mec='#8E2A0B', mew=1.2, label=f'{int(h28.n.max())} seasons (1997–2026)'))
    handles.append(Line2D([0], [0], color=BENCH, lw=1.5, ls=(0, (6, 3)), label=f'No temperature ({bench[loc]:.1f} days)'))
    leg = ax.legend(handles=handles, loc='upper left', frameon=True, facecolor='white', edgecolor='none',
                    framealpha=1.0, fontsize=8.2, handlelength=2.4, borderaxespad=0.2, labelspacing=0.45)
    leg.set_zorder(10)
axes[0].set_ylabel('MAE of the onset (days)')
fig.tight_layout(w_pad=1.2)
fig.savefig(OUT_DIR / 'fig_mae_vs_tbase.png', dpi=300, facecolor='white', bbox_inches='tight')
plt.show()

## Step 12: Save
Save the summary, the yearly results and the onsets as csv files.

In [ ]:
summary.to_csv(OUT_DIR / 'tbase_loyo_summary.csv', index=False)
loyo.to_csv(OUT_DIR / 'tbase_loyo_per_year.csv', index=False)
onsets.to_csv(OUT_DIR / 'onsets_per_year.csv', index=False)